# Attribution Analysis — AlphaGenome FT ChIP-seq (C. elegans)

Compute and visualize attributions for fine-tuned AlphaGenome ChIP models on MYRF-1 motif hit regions.

**Methods available:**
- **Gradient SHAP**: Expected gradients with Altschul-Erickson dinucleotide shuffle (`deepshap.py`)
- **Gradient x Input**: Standard gradient-based
- **ISM**: In silico mutagenesis (wildtype-base importance)

In [1]:
import os, sys, json
import numpy as np
import jax
import jax.numpy as jnp
from pathlib import Path

# Project paths
PROJ_DIR = Path("/grid/wsbs/home_norepl/pmantill/Worm_ChiP")
RESULTS_DIR = PROJ_DIR / "results"
DATA_DIR = PROJ_DIR / "data"
GENOME_FA = str(PROJ_DIR / "genome" / "ce11.fa")

# Add src modules to path (EncoderChIPHead, ChIPDataset)
sys.path.insert(0, str(PROJ_DIR))
# Add interp scripts to path (deepshap module)
sys.path.insert(0, str(PROJ_DIR / "interp"))

# Fix alphagenome_ft namespace shadowing: the alphagenome_ft/ git clone dir
# in the project root shadows the editable-installed package. Force the real
# package path to win.
_aft_real_parent = str(PROJ_DIR / "alphagenome_ft")
if _aft_real_parent not in sys.path:
    sys.path.insert(0, _aft_real_parent)
if "alphagenome_ft" in sys.modules:
    _mod = sys.modules["alphagenome_ft"]
    if not hasattr(_mod, "CustomHead"):
        del sys.modules["alphagenome_ft"]

# HuggingFace cache
os.environ["HF_HOME"] = os.path.expanduser("~/Liver_AGFT/.weights/huggingface")

# Patch: use HuggingFace instead of Kaggle for base model downloads
from alphagenome_research.model import dna_model
dna_model.create_from_kaggle = dna_model.create_from_huggingface

from deepshap import deep_lift_shap

print(f"JAX devices: {jax.devices()}")

/grid/wsbs/home_norepl/pmantill/Worm_ChiP/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1773091412.583673 3286000 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773091428.055920 3286000 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


JAX devices: [CudaDevice(id=0)]


## 1. Configuration

Set the model name, target, and attribution parameters.

In [2]:
# ---- Configuration ----
MODEL_NAME = "chip_gfp"                  # Model to analyze (chip_gfp, chip_polii, gfp_do03_gfp, etc.)
CHECKPOINT_STAGE = "best_stage2"          # "best" (stage 1) or "best_stage2"
HEAD_NAME = "chip_head"
WINDOW_SIZE = 1000                        # bp window around each motif hit

# Attribution parameters
ATTRIBUTION_METHOD = "deepshap"           # "deepshap", "gradient_x_input", or "ism"
N_SEQUENCES = 10                          # Number of top motif hits to analyze
N_SHAP_REFERENCES = 20                    # Number of shuffle references for DeepSHAP
N_IG_STEPS = 50                           # Integration steps per reference

# Motif hits file
MOTIF_HITS_FILE = str(DATA_DIR / "MYRF-1_motif_hits_2.gff.txt")

# BigWig paths
BIGWIG_GFP = str(DATA_DIR / "DR16-GFP-1Aligned.sortedByCoord_removeDup.out.bw")
BIGWIG_POLII = str(DATA_DIR / "DR17-PolII-1Aligned.sortedByCoord_removeDup.out.bw")

# Paths
model_dir = RESULTS_DIR / MODEL_NAME
checkpoint_dir = model_dir / "checkpoints" / CHECKPOINT_STAGE
output_dir = PROJ_DIR / "interp" / "results" / MODEL_NAME
output_dir.mkdir(parents=True, exist_ok=True)

# Load model hyperparameters
with open(model_dir / "args.json") as f:
    args_json = json.load(f)
hp = args_json["hp"]

print(f"Model: {MODEL_NAME}")
print(f"Checkpoint: {CHECKPOINT_STAGE}")
print(f"DeepSHAP: {N_SHAP_REFERENCES} refs x {N_IG_STEPS} IG steps = {N_SHAP_REFERENCES * N_IG_STEPS} grad evals")
print(f"Head config: pooling={hp['pooling_type']}, nl={hp['nl_size']}, do={hp['dropout']}, act={hp['activation']}")

Model: chip_gfp
Checkpoint: best_stage2
DeepSHAP: 20 refs x 50 IG steps = 1000 grad evals
Head config: pooling=mean, nl=1024, do=0.1, act=relu


## 2. Load Model

In [3]:
from alphagenome.models import dna_output
from alphagenome_ft import (
    register_custom_head,
    load_checkpoint,
    CustomHeadConfig,
    CustomHeadType,
)
from src.chip_heads import EncoderChIPHead

# Register head with same config as training
nl_size = hp["nl_size"] if isinstance(hp["nl_size"], list) else [hp["nl_size"]]
head_metadata = {
    "pooling_type": hp["pooling_type"],
    "nl_size": nl_size,
    "do": hp["dropout"],
    "activation": hp["activation"],
}
register_custom_head(
    HEAD_NAME, EncoderChIPHead,
    CustomHeadConfig(
        type=CustomHeadType.GENOME_TRACKS,
        output_type=dna_output.OutputType.RNA_SEQ,
        num_tracks=1, metadata=head_metadata,
    ),
)

# Load fine-tuned checkpoint
model = load_checkpoint(
    str(checkpoint_dir),
    base_model_version="all_folds",
    init_seq_len=WINDOW_SIZE,
)
print(f"Model loaded from {checkpoint_dir}")
print(f"Custom heads: {model._custom_heads}")
print(f"Total parameters: {model.count_parameters():,}")

Loading checkpoint from /grid/wsbs/home_norepl/pmantill/Worm_ChiP/results/chip_gfp/checkpoints/best_stage2
  Custom heads: ['chip_head']
  Model type: Full model
  Inferred use_encoder_output=True from parameter structure (flat head keys detected)
Loading full model from checkpoint...


Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 106634.85it/s]
/grid/wsbs/home_norepl/pmantill/Worm_ChiP/.venv/lib/python3.11/site-packages/pyfaidx/__init__.py:589: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
/grid/wsbs/home_norepl/pmantill/Worm_ChiP/.venv/lib/python3.11/site-packages/pyfaidx/__init__.py:589: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 115704.94it/s]
/grid/wsbs/home_norepl/pmantill/Worm_ChiP/.venv/lib/python3.11/site-packages/pyfaidx/__init__.py:589: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
/grid/wsbs/home_norepl/pmantill/Worm_ChiP/.venv/lib/python3.11/site-packages/pyfaidx/__init__.py

✓ Checkpoint loaded successfully
  Total parameters: 452,030,598
Model loaded from /grid/wsbs/home_norepl/pmantill/Worm_ChiP/results/chip_gfp/checkpoints/best_stage2
Custom heads: ['chip_head']
Total parameters: 452,030,598


## 3. Load MYRF-1 Motif Hits

Parse the GFF file and extract windows centered on top motif hits (ranked by FIMO score).

In [4]:
import re
from pysam import FastaFile
import pyBigWig

def parse_gff_hits(gff_path):
    """Parse FIMO GFF motif hits file. Returns list of dicts sorted by score (descending)."""
    hits = []
    with open(gff_path) as f:
        for line in f:
            if line.startswith("#"):
                continue
            parts = line.strip().split("\t")
            if len(parts) < 9:
                continue
            chrom = parts[0]
            start = int(parts[3]) - 1  # GFF is 1-based -> 0-based
            end = int(parts[4])         # GFF end is inclusive, but we keep as-is for center calc
            score = float(parts[5])
            strand = parts[6]
            # Parse attributes for sequence
            attrs = parts[8]
            seq_match = re.search(r'sequence=([ACGTacgt]+)', attrs)
            pval_match = re.search(r'pvalue=([\d.e-]+)', attrs)
            motif_seq = seq_match.group(1) if seq_match else ""
            pvalue = float(pval_match.group(1)) if pval_match else 1.0
            hits.append({
                "chrom": chrom, "start": start, "end": end,
                "score": score, "strand": strand,
                "motif_seq": motif_seq, "pvalue": pvalue,
            })
    hits.sort(key=lambda x: x["score"], reverse=True)
    return hits

hits = parse_gff_hits(MOTIF_HITS_FILE)
print(f"Total motif hits: {len(hits)}")
print(f"\nTop {N_SEQUENCES} hits by FIMO score:")
for i, h in enumerate(hits[:N_SEQUENCES], 1):
    print(f"  {i}. {h['chrom']}:{h['start']}-{h['end']} ({h['strand']}) "
          f"score={h['score']:.1f} pval={h['pvalue']:.2e} seq={h['motif_seq']}")

Total motif hits: 53219

Top 10 hits by FIMO score:
  1. chrI:721265-721284 (-) score=130.0 pval=1.05e-13 seq=GCGGTGGAGCGCGCTTGCA
  2. chrI:12486924-12486943 (+) score=130.0 pval=1.05e-13 seq=GCGGTGGAGCGCGCTTGCA
  3. chrIII:1609909-1609928 (+) score=130.0 pval=1.05e-13 seq=GCGGTGGAGCGCGCTTGCA
  4. chrIII:3456425-3456444 (-) score=130.0 pval=1.05e-13 seq=GCGGTGGAGCGCGCTTGCA
  5. chrIV:17012139-17012158 (-) score=130.0 pval=1.05e-13 seq=GCGGTGGAGCGCGCTTGCA
  6. chrII:12099202-12099221 (+) score=125.0 pval=2.98e-13 seq=GCGATGGAGCGCGCTTGCA
  7. chrI:405779-405798 (-) score=123.0 pval=4.90e-13 seq=TCGGTGGAGCGCGCTTGCA
  8. chrI:500417-500436 (+) score=123.0 pval=4.90e-13 seq=tcggtggagcgcgcttgca
  9. chrI:862549-862568 (+) score=123.0 pval=4.90e-13 seq=tcggtggagcgcgcttgca
  10. chrI:1174978-1174997 (-) score=123.0 pval=4.90e-13 seq=TCGGTGGAGCGCGCTTGCA


In [5]:
def extract_window(fasta_path, chrom, motif_start, motif_end, window_size=1000):
    """Extract a window_size bp window centered on the motif hit.
    Returns one-hot encoded sequence (1, window_size, 4) and genomic coords."""
    fasta = FastaFile(fasta_path)
    chrom_len = fasta.get_reference_length(chrom)

    motif_center = (motif_start + motif_end) // 2
    win_start = motif_center - window_size // 2
    win_end = win_start + window_size

    # Clamp to chromosome bounds
    if win_start < 0:
        win_start = 0
        win_end = window_size
    if win_end > chrom_len:
        win_end = chrom_len
        win_start = chrom_len - window_size

    seq_str = fasta.fetch(chrom, win_start, win_end)
    fasta.close()

    # One-hot encode (ACGT = 0,1,2,3)
    base_map = {"A": 0, "C": 1, "G": 2, "T": 3, "a": 0, "c": 1, "g": 2, "t": 3}
    ohe = np.zeros((window_size, 4), dtype=np.float32)
    for i, base in enumerate(seq_str):
        idx = base_map.get(base)
        if idx is not None:
            ohe[i, idx] = 1.0

    seq_jax = jnp.array(ohe[None, :, :])  # (1, window_size, 4)
    org = jnp.array([0])  # organism_index (human idx; AG has no worm)
    return seq_jax, org, seq_str, (chrom, win_start, win_end)


def get_bigwig_coverage(bw_path, chrom, start, end):
    """Get total coverage from BigWig over a region."""
    bw = pyBigWig.open(bw_path)
    vals = bw.values(chrom, start, end)
    bw.close()
    return float(np.nansum(vals))


# Prepare sequences for top hits
top_hits = hits[:N_SEQUENCES]
print(f"Extracting {WINDOW_SIZE}bp windows centered on top {N_SEQUENCES} motif hits...")
for i, h in enumerate(top_hits, 1):
    seq, org, seq_str, (chrom, ws, we) = extract_window(
        GENOME_FA, h["chrom"], h["start"], h["end"], WINDOW_SIZE
    )
    gfp_cov = get_bigwig_coverage(BIGWIG_GFP, chrom, ws, we)
    polii_cov = get_bigwig_coverage(BIGWIG_POLII, chrom, ws, we)
    print(f"  {i}. {chrom}:{ws}-{we} | GFP cov={gfp_cov:.1f} PolII cov={polii_cov:.1f}")

Extracting 1000bp windows centered on top 10 motif hits...


  1. chrI:720774-721774 | GFP cov=1407.7 PolII cov=1082.0
  2. chrI:12486433-12487433 | GFP cov=1782.9 PolII cov=1311.3
  3. chrIII:1609418-1610418 | GFP cov=1233.8 PolII cov=1246.6
  4. chrIII:3455934-3456934 | GFP cov=4492.7 PolII cov=2589.7
  5. chrIV:17011648-17012648 | GFP cov=1019.8 PolII cov=1191.1
  6. chrII:12098711-12099711 | GFP cov=1367.1 PolII cov=1645.8
  7. chrI:405288-406288 | GFP cov=1344.8 PolII cov=1255.2
  8. chrI:499926-500926 | GFP cov=1222.3 PolII cov=669.4
  9. chrI:862058-863058 | GFP cov=1443.4 PolII cov=688.8
  10. chrI:1174487-1175487 | GFP cov=1377.5 PolII cov=1135.7


## 4. Helper Functions

In [6]:
def decode_one_hot(one_hot_seq):
    """Convert one-hot encoded sequence back to DNA string (A=0, C=1, G=2, T=3)."""
    if one_hot_seq.ndim == 3:
        one_hot_seq = one_hot_seq[0]
    base_map = {0: 'A', 1: 'C', 2: 'G', 3: 'T'}
    return ''.join(base_map[int(jnp.argmax(one_hot_seq[i]))] for i in range(one_hot_seq.shape[0]))

## 5. Quick Prediction Check

Verify the model produces reasonable predictions on the top motif hit.

In [7]:
from deepshap import _build_compute_output

h = top_hits[0]
seq, org, seq_str, (chrom, ws, we) = extract_window(
    GENOME_FA, h["chrom"], h["start"], h["end"], WINDOW_SIZE
)

compute_output = _build_compute_output(model, org, HEAD_NAME)
pred = float(compute_output(seq))
gfp_cov = get_bigwig_coverage(BIGWIG_GFP, chrom, ws, we)

print(f"Top hit: {h['chrom']}:{h['start']}-{h['end']}")
print(f"  Model prediction (log1p scale): {pred:.4f}")
print(f"  Actual GFP coverage (raw sum):  {gfp_cov:.1f}")
print(f"  Actual log1p(coverage):         {np.log1p(gfp_cov):.4f}")

Top hit: chrI:721265-721284
  Model prediction (log1p scale): 57.5000
  Actual GFP coverage (raw sum):  1407.7
  Actual log1p(coverage):         7.2504


## 6. Compute Attributions & Visualize

Change `HIT_INDEX` to analyze a different motif hit (0-indexed into `top_hits`).

In [8]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

# ---- Change this index to analyze a different hit ----
HIT_INDEX = 0

h = top_hits[HIT_INDEX]
rank = HIT_INDEX + 1
seq, org, seq_str, (chrom, ws, we) = extract_window(
    GENOME_FA, h["chrom"], h["start"], h["end"], WINDOW_SIZE
)
motif_label = f"{h['chrom']}:{h['start']}-{h['end']}({h['strand']})"

# Motif position within the window (for annotation)
motif_start_in_win = h["start"] - ws
motif_end_in_win = h["end"] - ws

print(f"Hit {rank}/{len(top_hits)}: {motif_label} score={h['score']:.1f} pval={h['pvalue']:.2e}")
print(f"Window: {chrom}:{ws}-{we} ({WINDOW_SIZE}bp)")
print(f"Motif at positions {motif_start_in_win}-{motif_end_in_win} within window")

methods = {
    "Grad x Input": lambda: model.compute_input_gradients(
        sequence=seq, organism_index=org, head_name=HEAD_NAME, gradients_x_input=True,
    ),
    "ISM": lambda: model.compute_ism_attributions(
        sequence=seq, organism_index=org, head_name=HEAD_NAME,
    ),
}

hit_out = output_dir / f"hit_{rank}_{h['chrom']}_{h['start']}"
hit_out.mkdir(parents=True, exist_ok=True)

for name, compute_fn in methods.items():
    attr = jnp.float32(compute_fn())
    if name != "ISM":
        attr = attr - jnp.mean(attr, axis=-1, keepdims=True)

    for mode, suffix in [("full", ""), ("wt", "_wt")]:
        grad = attr * seq if mode == "wt" else attr
        mask = mode == "wt"
        tag = f" (WT only)" if mode == "wt" else ""
        save_path = hit_out / f"{name.replace(' ', '_').lower()}{suffix}.png"

        model.plot_sequence_logo(
            sequence=seq, gradients=grad, save_path=str(save_path),
            logo_type="weight", mask_to_sequence=mask, use_absolute=False,
            title=f"{name}{tag} — {motif_label}",
        )

        # Re-open saved figure to annotate motif region
        img = plt.imread(str(save_path))
        fig, ax = plt.subplots(figsize=(img.shape[1] / 100, img.shape[0] / 100), dpi=100)
        ax.imshow(img)
        ax.axis("off")

        # Draw motif region box (map bp positions to image x-coords)
        # Logo plot x-axis spans the full window; image pixel columns map linearly
        img_w = img.shape[1]
        # Approximate: logo area is roughly 10%-90% of image width (matplotlib margins)
        left_margin = 0.10 * img_w
        right_margin = 0.98 * img_w
        plot_w = right_margin - left_margin
        x0 = left_margin + (motif_start_in_win / WINDOW_SIZE) * plot_w
        x1 = left_margin + (motif_end_in_win / WINDOW_SIZE) * plot_w
        rect = mpatches.FancyBboxPatch(
            (x0, 0), x1 - x0, img.shape[0],
            boxstyle="square,pad=0", linewidth=0,
            edgecolor="none", facecolor="red", alpha=0.12,
        )
        ax.add_patch(rect)
        # Red lines at motif boundaries
        ax.axvline(x0, color="red", linewidth=1.5, linestyle="--", alpha=0.7)
        ax.axvline(x1, color="red", linewidth=1.5, linestyle="--", alpha=0.7)

        fig.savefig(str(save_path), bbox_inches="tight", dpi=150, pad_inches=0.02)
        plt.close(fig)

    print(f"  Computed {name}")

print(f"Saved to: {hit_out}")

Hit 1/10: chrI:721265-721284(-) score=130.0 pval=1.05e-13
Window: chrI:720774-721774 (1000bp)
Motif at positions 491-510 within window


E0309 17:24:57.528417 3287090 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


Sequence logo saved to: /grid/wsbs/home_norepl/pmantill/Worm_ChiP/interp/results/chip_gfp/hit_1_chrI_721265/grad_x_input.png
Sequence logo saved to: /grid/wsbs/home_norepl/pmantill/Worm_ChiP/interp/results/chip_gfp/hit_1_chrI_721265/grad_x_input_wt.png
  Computed Grad x Input
Sequence logo saved to: /grid/wsbs/home_norepl/pmantill/Worm_ChiP/interp/results/chip_gfp/hit_1_chrI_721265/ism.png
Sequence logo saved to: /grid/wsbs/home_norepl/pmantill/Worm_ChiP/interp/results/chip_gfp/hit_1_chrI_721265/ism_wt.png
  Computed ISM
Saved to: /grid/wsbs/home_norepl/pmantill/Worm_ChiP/interp/results/chip_gfp/hit_1_chrI_721265


## 7. Zoom into Motif Region

Visualize attributions zoomed to the motif +/- flanking bases for the top hit.

In [10]:
# Zoom into the motif region (+/- 50bp flank) for the current hit
h = top_hits[HIT_INDEX]
seq, org, seq_str, (chrom, ws, we) = extract_window(
    GENOME_FA, h["chrom"], h["start"], h["end"], WINDOW_SIZE
)

motif_start_in_win = h["start"] - ws
motif_end_in_win = h["end"] - ws

FLANK = 50
zoom_start = max(0, motif_start_in_win - FLANK)
zoom_end = min(WINDOW_SIZE, motif_end_in_win + FLANK)
zoom_len = zoom_end - zoom_start

# Motif position within the zoomed region
motif_s_zoom = motif_start_in_win - zoom_start
motif_e_zoom = motif_end_in_win - zoom_start

seq_zoom = seq[:, zoom_start:zoom_end, :]

hit_out = output_dir / f"hit_{HIT_INDEX+1}_{h['chrom']}_{h['start']}"
hit_out.mkdir(parents=True, exist_ok=True)

zoom_methods = {
    "Grad x Input": model.compute_input_gradients(
        sequence=seq, organism_index=org, head_name=HEAD_NAME, gradients_x_input=True,
    ),
    "ISM": model.compute_ism_attributions(
        sequence=seq, organism_index=org, head_name=HEAD_NAME,
    ),
}

for name, attr_full in zoom_methods.items():
    attr_full = jnp.float32(attr_full)
    if name != "ISM":
        attr_full = attr_full - jnp.mean(attr_full, axis=-1, keepdims=True)
    attr_zoom = attr_full[:, zoom_start:zoom_end, :]

    save_path = hit_out / f"{name.replace(' ', '_').lower()}_zoom.png"
    model.plot_sequence_logo(
        sequence=seq_zoom, gradients=attr_zoom, save_path=str(save_path),
        logo_type="weight", mask_to_sequence=False, use_absolute=False,
        title=f"{name} zoom — {h['chrom']}:{h['start']}-{h['end']} +/- {FLANK}bp",
    )

    # Annotate motif region
    img = plt.imread(str(save_path))
    fig, ax = plt.subplots(figsize=(img.shape[1] / 100, img.shape[0] / 100), dpi=100)
    ax.imshow(img)
    ax.axis("off")
    img_w = img.shape[1]
    left_margin = 0.10 * img_w
    right_margin = 0.98 * img_w
    plot_w = right_margin - left_margin
    x0 = left_margin + (motif_s_zoom / zoom_len) * plot_w
    x1 = left_margin + (motif_e_zoom / zoom_len) * plot_w
    rect = mpatches.FancyBboxPatch(
        (x0, 0), x1 - x0, img.shape[0],
        boxstyle="square,pad=0", linewidth=0,
        edgecolor="none", facecolor="red", alpha=0.12,
    )
    ax.add_patch(rect)
    ax.axvline(x0, color="red", linewidth=1.5, linestyle="--", alpha=0.7)
    ax.axvline(x1, color="red", linewidth=1.5, linestyle="--", alpha=0.7)
    fig.savefig(str(save_path), bbox_inches="tight", dpi=150, pad_inches=0.02)
    plt.close(fig)

    print(f"  {name} zoom saved: {save_path}")

Sequence logo saved to: /grid/wsbs/home_norepl/pmantill/Worm_ChiP/interp/results/chip_gfp/hit_1_chrI_721265/grad_x_input_zoom.png
  Grad x Input zoom saved: /grid/wsbs/home_norepl/pmantill/Worm_ChiP/interp/results/chip_gfp/hit_1_chrI_721265/grad_x_input_zoom.png
Sequence logo saved to: /grid/wsbs/home_norepl/pmantill/Worm_ChiP/interp/results/chip_gfp/hit_1_chrI_721265/ism_zoom.png
  ISM zoom saved: /grid/wsbs/home_norepl/pmantill/Worm_ChiP/interp/results/chip_gfp/hit_1_chrI_721265/ism_zoom.png
